In [ ]:
from project_config import WEIGHTS_ROOT, OUTPUT_ROOT, TRAIN_SPLIT, VALIDATION_SPLIT, RANDOM_SEED, NUM_WORKERS, DEVICES, STRATEGY, load_split_samples
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import numpy as np
import os
from pytorch_lightning.callbacks import TQDMProgressBar, ModelCheckpoint


In [ ]:
pl.seed_everything(RANDOM_SEED, workers=True)
train_dic_x, train_dic_mask, train_dic_y = load_split_samples(TRAIN_SPLIT)
val_dic_x, val_dic_mask, val_dic_y = load_split_samples(VALIDATION_SPLIT)
with np.load(train_dic_x[0]) as sample:
    in_channels = sample['arr_0'].shape[0]
with np.load(train_dic_y[0]) as sample:
    out_channels = sample['arr_0'].shape[0]


In [ ]:
class TDconv2d_layer(nn.Module):

    def __init__(self, inchannels):
        super(TDconv2d_layer, self).__init__()
        self.c = inchannels
        self.act = nn.LeakyReLU(inplace=True)
        self.tdconv = nn.Sequential(nn.Conv2d(self.c, self.c, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(self.c), self.act, nn.Conv2d(self.c, self.c, kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(self.c))

    def forward(self, x):
        identity = x
        out = self.tdconv(x)
        out = 0.3 * out + identity
        return self.act(out)

class HRUnet2D(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(HRUnet2D, self).__init__()
        chans = [128, 256, 512]
        ks = 3
        self.act = nn.LeakyReLU(inplace=True)
        self.act_output = nn.LeakyReLU(inplace=True)
        self.start_conv = nn.Sequential(nn.Conv2d(in_channels, chans[0], kernel_size=3, stride=1, padding=1, bias=False), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_11 = TDconv2d_layer(chans[0])
        self.ResBlock_12 = TDconv2d_layer(chans[0])
        self.DownConv_1 = nn.Sequential(nn.MaxPool2d(kernel_size=2, stride=2), nn.Conv2d(chans[0], chans[1], kernel_size=1, stride=1, bias=False), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_21 = TDconv2d_layer(chans[1])
        self.ResBlock_22 = TDconv2d_layer(chans[1])
        self.DownConv_2 = nn.Sequential(nn.MaxPool2d(kernel_size=2, stride=2), nn.Conv2d(chans[1], chans[2], kernel_size=1, stride=1, bias=False), nn.BatchNorm2d(chans[2]), self.act)
        self.ResBlock_31 = TDconv2d_layer(chans[2])
        self.ResBlock_32 = TDconv2d_layer(chans[2])
        self.ResBlock_33 = TDconv2d_layer(chans[2])
        self.ResBlock_34 = TDconv2d_layer(chans[2])
        self.UpConv_1 = nn.ConvTranspose2d(chans[2], chans[1], kernel_size=2, stride=2)
        self.UpConv_next_1 = nn.Sequential(nn.Conv2d(chans[1] * 2, chans[1], kernel_size=ks, stride=1, padding=1, bias=False), nn.BatchNorm2d(chans[1]), self.act)
        self.ResBlock_23 = TDconv2d_layer(chans[1])
        self.ResBlock_24 = TDconv2d_layer(chans[1])
        self.UpConv_2 = nn.ConvTranspose2d(chans[1], chans[0], kernel_size=2, stride=2)
        self.UpConv_next_2 = nn.Sequential(nn.Conv2d(chans[0] * 2, chans[0], kernel_size=ks, stride=1, padding=1, bias=False), nn.BatchNorm2d(chans[0]), self.act)
        self.ResBlock_13 = TDconv2d_layer(chans[0])
        self.ResBlock_14 = TDconv2d_layer(chans[0])
        self.final_conv = nn.Sequential(nn.Conv2d(chans[0], out_channels, kernel_size=3, stride=1, padding=1), self.act_output)

    def _pad_to_match(self, x, skip_x):
        diffY = skip_x.size()[2] - x.size()[2]
        diffX = skip_x.size()[3] - x.size()[3]
        return F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])

    def forward(self, x, mask=None):
        x = self.start_conv(x)
        x = self.ResBlock_11(x)
        x = self.ResBlock_12(x)
        x_pass_1 = x
        x = self.DownConv_1(x)
        x = self.ResBlock_21(x)
        x = self.ResBlock_22(x)
        x_pass_2 = x
        x = self.DownConv_2(x)
        x = self.ResBlock_31(x)
        x = self.ResBlock_32(x)
        x = self.ResBlock_33(x)
        x = self.ResBlock_34(x)
        x = self.UpConv_1(x)
        x = self._pad_to_match(x, x_pass_2)
        x = torch.cat((x, x_pass_2), dim=1)
        x = self.UpConv_next_1(x)
        x = self.ResBlock_23(x)
        x = self.ResBlock_24(x)
        x = self.UpConv_2(x)
        x = self._pad_to_match(x, x_pass_1)
        x = torch.cat((x, x_pass_1), dim=1)
        x = self.UpConv_next_2(x)
        x = self.ResBlock_13(x)
        x = self.ResBlock_14(x)
        x = self.final_conv(x)
        if mask is not None:
            x = x * mask
        return x


In [ ]:
class SequenceDataset(Dataset):

    def __init__(self, file_dic_x_radar, file_dic_x_mask, file_dic_y):
        self.file_dic_x_radar = file_dic_x_radar
        self.file_dic_x_mask = file_dic_x_mask
        self.file_dic_y = file_dic_y

    def __len__(self):
        return len(self.file_dic_x_radar)

    def __getitem__(self, idx):
        input_data = np.load(self.file_dic_x_radar[idx])['arr_0']
        mask_data = np.load(self.file_dic_x_mask[idx])['arr_0']
        output_data = np.load(self.file_dic_y[idx])['arr_0']
        input_tensor = torch.from_numpy(input_data).float()
        mask_tensor = torch.from_numpy(mask_data).float().unsqueeze(0)
        output_tensor = torch.from_numpy(output_data).float()
        return (input_tensor, mask_tensor, output_tensor)

class SequenceDataModule(pl.LightningDataModule):

    def __init__(self, batch_size, num_workers):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_dataset = SequenceDataset(train_dic_x, train_dic_mask, train_dic_y)
        self.val_dataset = SequenceDataset(val_dic_x, val_dic_mask, val_dic_y)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True, persistent_workers=self.num_workers > 0, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True, persistent_workers=self.num_workers > 0)

class Loss(nn.Module):

    def __init__(self, alpha, beta):
        super(Loss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        base_temp = torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        if torch.sum(y_true) > 0:
            temp = base_temp
        else:
            temp = self.beta * base_temp
        divisor = temp.shape[1] * temp.shape[2] * temp.shape[3]
        return torch.sum(temp) / divisor

class LightningModel(pl.LightningModule):

    def __init__(self, alpha, beta):
        super().__init__()
        self.model = HRUnet2D(in_channels=in_channels, out_channels=out_channels)
        self.criterion = Loss(alpha, beta)

    def forward(self, x, mask):
        return self.model(x, mask)

    def training_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, mask, labels = batch
        outputs = self.model(inputs, mask)
        val_loss = self.criterion(outputs, labels)
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.0001, fused=torch.cuda.is_available())
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}

class SilentValidationProgressBar(TQDMProgressBar):

    def init_validation_tqdm(self):
        bar = super().init_validation_tqdm()
        bar.disable = True
        return bar


In [ ]:
dirpath = WEIGHTS_ROOT / 'UNet'
if not os.path.exists(dirpath):
    os.makedirs(dirpath)
checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=3, filename='best-model-{epoch:02d}-{val_loss:.8f}', dirpath=dirpath, save_weights_only=False)
trainer = pl.Trainer(accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=DEVICES, precision='16-mixed' if torch.cuda.is_available() else '32-true', strategy=STRATEGY, max_epochs=100, callbacks=[SilentValidationProgressBar(), checkpoint_callback], benchmark=True)
model = LightningModel(0.1, 10)
datamodule = SequenceDataModule(batch_size=8, num_workers=NUM_WORKERS)
trainer.fit(model, datamodule=datamodule)


In [ ]:
trainer.save_checkpoint(str(dirpath / 'last.ckpt'))
